# DrenaVision 0.4 — reprodução do detector v3 no Colab

Este notebook prepara o conjunto `v3_clean` e repete o fine-tuning iniciado do checkpoint v2 preservado. O produto também usa o SegFormer B1 para água. Sua configuração publicada foi escolhida na validação; este notebook não recalibra os parâmetros no teste.

**Outputs vazios: validado por sintaxe/Ruff, não executado no Colab nesta entrega.** A execução publicada ocorreu em Windows/RTX 4060: limite de 80 épocas, parada na 63 e melhor época 43. Isso não garante pesos idênticos em outra plataforma.

Ative uma GPU em **Ambiente de execução → Alterar tipo de ambiente de execução**. O modelo NVIDIA é licenciado para pesquisa/avaliação não comercial; imagens iRodent também exigem observar a condição CC BY-NC adotada pelo projeto.


## 1. Obter o checkout autorizado

Preencha a URL do repositório. Um repositório privado exige acesso autorizado. Não escreva tokens em células, outputs ou URLs compartilhadas.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

REPO_URL = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
PROJECT = Path("/content/drenavision")
if not (PROJECT / "pyproject.toml").is_file():
    if not REPO_URL.strip():
        raise ValueError("Preencha REPO_URL com o checkout autorizado.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)],
        check=True,
    )
os.chdir(PROJECT)
print(
    subprocess.run(
        ["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True
    ).stdout.strip()
)

## 2. Instalar as versões testadas e conferir a GPU

A instalação pode exigir reiniciar o runtime se bibliotecas antigas já estiverem carregadas. Os wheels CUDA específicos do Windows não são usados no Colab.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt"], check=True
)

In [ ]:
import torch

print(
    {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
)
DEVICE = "0" if torch.cuda.is_available() else "cpu"
if not torch.cuda.is_available():
    print("Treino em CPU pode demorar; confira o runtime selecionado.")

## 3. Persistência opcional no Drive

O treino continua em `/content`; copie os resultados ao final para não perdê-los quando a sessão expirar.


In [ ]:
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT = Path("/content/drive/MyDrive/drenavision-artifacts")
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

## 4. Fornecer os assets exatos da release

Obtenha `best_v2.pt` e `exemplos_reais.zip` na [release v0.4.0-pipeline-v3](https://github.com/Torugo0/drenahub-drenavision/releases/tag/v0.4.0-pipeline-v3), com sua conta autorizada, e disponibilize-os em `/content` ou no Drive. Ajuste os dois caminhos abaixo.

O checkpoint v2 é o **inicial do treino**, não o ativo v3. O ZIP de exemplos preserva os MP4 anotados: reencodificar com outro FFmpeg pode mudar hashes/pixels e não reproduzir os quadros revisados. A verificação recusa fontes divergentes.


In [ ]:
BEST_V2_SOURCE = "/content/best_v2.pt"  # @param {type:"string"}
EXAMPLES_ZIP_SOURCE = "/content/exemplos_reais.zip"  # @param {type:"string"}
models = Path("resultados/local/models")
models.mkdir(parents=True, exist_ok=True)
initial = models / "best_v2.pt"
if not initial.is_file():
    source = Path(BEST_V2_SOURCE)
    if not source.is_file():
        raise FileNotFoundError(
            "Forneça best_v2.pt da release; consulte REPRODUZIR.md."
        )
    shutil.copy2(source, initial)
expected = "01ea7c578783b2a08fc585051b0768a1e9c0d510cb5652173fab0979f8d7d27a"
if hashlib.sha256(initial.read_bytes()).hexdigest() != expected:
    raise ValueError("O checkpoint inicial não corresponde ao v2 preservado.")
archive_source = Path(EXAMPLES_ZIP_SOURCE)
if not archive_source.is_file():
    raise FileNotFoundError("Forneça exemplos_reais.zip da release.")
examples = Path("dados/examples").resolve()
examples.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_source) as bundle:
    if "sources.json" not in bundle.namelist():
        raise ValueError("O ZIP deve ter sources.json e mídias na raiz.")
    for member in bundle.infolist():
        (examples / member.filename).resolve().relative_to(examples)
    if bundle.testzip() is not None:
        raise ValueError("Falha de integridade do ZIP de exemplos.")
    bundle.extractall(examples)
print("Inicial v2 verificado; exemplos exatos extraídos.")

## 5. Baixar as fontes e reconstruir a base v2

CLICK usa HTTP Range para a amostra 100/20/20, sem baixar seu ZIP completo. O v2 recupera os brutos pelo manifesto v1 versionado; não precisa reconstruir o dataset com caixas defeituosas. Seus arquivos são preservados como base da revisão seguinte.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/download_datasets.py",
        "--source",
        "all",
        "--seed",
        "42",
    ],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [sys.executable, "ferramentas/scripts/prepare_negative_crops.py"],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [sys.executable, "ferramentas/scripts/prepare_domain_annotations.py"],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/prepare_adaptation.py",
        "--domain-root",
        "dados/interim/adaptation",
    ],
    cwd=PROJECT,
    check=True,
)

## 6. Montar v3_clean com contexto e dados adicionais

São 400 imagens positivas de água adicionais, 150 novas imagens nativas de treino CLICK, cinco quadros completos de rato/resíduos e oito de água. Os recortes pequenos de rato são substituídos. Negativos de superfície molhada com rótulo vazio são excluídos por regra, porque podem conter poças. A montagem medida é 1.107/60/60; confira seu relatório.

As caixas de água conectadas pela interseção/contato original viram envelopes. Os envelopes podem conter espaços secos e não são máscaras. Dados de domínio são apenas de treino. Uma pasta preenchida não deve ser sobrescrita.


In [ ]:
subprocess.run(
    [sys.executable, "ferramentas/scripts/prepare_rat_context_v3.py"],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/prepare_water_v3.py",
        "--extra-negative",
        "0",
    ],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/prepare_v3_dataset.py",
        "--domain-root",
        "dados/interim/rat_context_v3",
        "--domain-root",
        "dados/interim/water_v3",
        "--fetch-click",
        "--output",
        "dados/processed/drenavision_v3_clean",
        "--audit-dir",
        "resultados/local/preparation_v3_clean",
    ],
    cwd=PROJECT,
    check=True,
)

## 7. Validar isolamento e conferir anotações

Nenhum treino deve continuar se a validação falhar. O pHash verifica proximidade visual entre partições; os relatórios registram ainda hashes exatos e grupos de origem.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/validate_dataset.py",
        "--phash",
        "--report",
        "resultados/local/preparation_v3_clean/validation_report.json",
    ],
    cwd=PROJECT,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/visualize_annotations.py",
        "--samples-per-split",
        "6",
        "--seed",
        "42",
    ],
    cwd=PROJECT,
    check=True,
)
from IPython.display import display
from PIL import Image

for preview in sorted(Path("resultados/local/dataset_preview_v3").rglob("*.jpg"))[:6]:
    print(preview)
    display(Image.open(preview))

## 8. Treinar o detector v3

O destino é `candidate_v3.pt`; o script não promove automaticamente o ativo. Máximo de 80 épocas, patience 20 e seed 42. Diminuir as épocas produz um teste técnico, não a reprodução do treino publicado. Não use `best.pt` como inicial nesta etapa.


In [ ]:
EPOCHS = 80  # @param {type:"integer"}
BATCH = 12  # @param {type:"integer"}
RUN_NAME = "yolo26n_water_context_v3_clean_colab"  # @param {type:"string"}
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/train.py",
        "--config",
        "ferramentas/configs/train_v3.yaml",
        "--epochs",
        str(EPOCHS),
        "--batch",
        str(BATCH),
        "--device",
        DEVICE,
        "--name",
        RUN_NAME,
    ],
    cwd=PROJECT,
    check=True,
)

## 9. Avaliar o candidato YOLO no mesmo teste v3

Esta avaliação é **somente do YOLO**, sem o segmentador B1 e os limiares por classe do produto. AP de validação escolhe a época; AP de teste é outro resultado. Use os mesmos rótulos v3 para comparar checkpoints. Não compare diretamente AP histórico v2 com AP v3, pois as caixas de água e a lista limpa mudaram. Após consultar o teste, não o use para reajustar parâmetros.


In [ ]:
EVAL_NAME = "candidate_v3_clean_test_colab"  # @param {type:"string"}
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/evaluate.py",
        "--config",
        "ferramentas/configs/evaluate_v3.yaml",
        "--model",
        "resultados/local/models/candidate_v3.pt",
        "--device",
        DEVICE,
        "--name",
        EVAL_NAME,
    ],
    cwd=PROJECT,
    check=True,
)
metric_files = sorted(
    Path("resultados/local/evaluation").rglob("metrics.json"),
    key=lambda p: p.stat().st_mtime,
)
if not metric_files:
    raise FileNotFoundError("A avaliação não gerou metrics.json.")
metrics_path = metric_files[-1]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Arquivo autoritativo:", metrics_path)
print(json.dumps(metrics["metrics"], indent=2, ensure_ascii=False))

## 10. Preparar o modelo fixo de água e executar uma análise real

O preparador baixa a revisão oficial NVIDIA fixada, verifica SHA-256 e converte somente os tensores em safetensors. Sua licença é de pesquisa/avaliação não comercial. O perfil publicado usa pixel 0,95, área mínima 5%, imagem YOLO 640 e animal 0,60. Não reotimize esses valores no teste desta sessão.

A inferência abaixo usa o candidato treinado nesta sessão com a água fixa. Seu hash/versão pode diferir do pipeline publicado. A referência de presença usada para calibrar água está versionada e é assistida, não certificação humana. O teste adicional de 119 imagens permanece separado.


In [ ]:
subprocess.run(
    [sys.executable, "ferramentas/scripts/prepare_water_model.py"],
    cwd=PROJECT,
    check=True,
)
test_images = [
    p
    for p in sorted(Path("dados/processed/drenavision_v3_clean/images/test").glob("*"))
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
if not test_images:
    raise FileNotFoundError("O split test não contém imagens.")
test_image = test_images[0]
subprocess.run(
    [
        sys.executable,
        "ferramentas/scripts/predict.py",
        "--input",
        str(test_image),
        "--model",
        "resultados/local/models/candidate_v3.pt",
        "--water-model",
        "resultados/local/models/water_segformer_b1_ade",
        "--water-pixel-confidence",
        "0.95",
        "--water-min-area-fraction",
        "0.05",
        "--animal-conf",
        "0.60",
        "--water-mode",
        "augment",
        "--imgsz",
        "640",
        "--device",
        DEVICE,
        "--output-dir",
        "resultados/local/predictions_colab",
    ],
    cwd=PROJECT,
    check=True,
)
annotated = (
    Path("resultados/local/predictions_colab") / f"{test_image.stem}_annotated.jpg"
)
display(Image.open(annotated))

## 11. Preservar pesos, relatórios e fingerprints

O pacote não inclui os dados brutos/processados completos. Salve fora da sessão antes de encerrá-la. Registre o commit, hardware, versões e o SHA-256 do novo checkpoint; só promova após avaliação e revisão comparativa.


In [ ]:
checkpoint = Path("resultados/local/models/candidate_v3.pt")
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
print("candidate_v3.pt SHA-256:", hashlib.sha256(checkpoint.read_bytes()).hexdigest())
bundle_root = Path(tempfile.mkdtemp(prefix="drenavision_evidence_v3_", dir="/content"))
for source in [
    Path("resultados/local/models"),
    Path("resultados/local/training"),
    Path("resultados/local/evaluation"),
    Path("resultados/local/preparation_v3_clean"),
    Path("resultados/local/predictions_colab"),
    Path("dados/processed/drenavision_v3_clean/data_manifest.csv"),
    Path("dados/processed/drenavision_v3_clean/dataset.yaml"),
    Path("ferramentas/configs"),
]:
    target = bundle_root / source
    if source.is_dir():
        shutil.copytree(source, target, dirs_exist_ok=True)
    elif source.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
archive = Path(shutil.make_archive(str(bundle_root), "zip", bundle_root))
print("Pacote:", archive, archive.stat().st_size, "bytes")
if USE_DRIVE:
    shutil.copy2(archive, DRIVE_OUTPUT / archive.name)
    print("Cópia no Drive:", DRIVE_OUTPUT / archive.name)

In [ ]:
DOWNLOAD_NOW = False  # @param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files

    files.download(str(archive))

## Encerramento

Confira se todas as células relevantes concluíram. Preserve `training_manifest.json`, `metrics.json`, configuração, manifesto dos dados e SHA-256. Não apresente um smoke test como treino final nem copie números sem vinculá-los ao checkpoint usado. O resultado de uma análise assistida não substitui inspeção humana do local.
